# Batch Inference Runner for 2D Cell Division Detection

This Jupyter notebook provides a template for running batch inference on 2D cell division detection using the DARE2d framework. It automates the process of applying multiple trained model pairs (regression and segmentation) to a set of input images.

**Author of this notebook: qazi**   

## Overview

This notebook performs the following tasks:
1. **Setup**: Configure paths and import necessary modules
2. **Data Discovery**: Identify input images for processing
3. **Batch Inference**: Run inference for each image with multiple model sets
4. **Postprocessing**: Aggregate multi-model results into consensus detections
5. **Quality Analysis**: Generate plots and metrics for result evaluation
6. **Results Summary**: Review all outputs and next steps

The pipeline processes 2D+t image stacks to detect cell division events using a multi-model consensus approach for improved robustness and uncertainty quantification.

## Requirements

- DARE2d project installed with all dependencies
- Input images in .tif format
- Pre-trained model checkpoints organized by model sets
- Python environment with required packages (see `requirements.txt`)

## Usage Instructions

1. Modify the path variables in the setup section to match your project structure
2. Place input images in the designated input directory
3. Ensure model checkpoints are available in the specified directories
4. Run the cells sequentially to execute the complete pipeline:
   - **Cell 2**: Setup and path configuration
   - **Cell 3**: Data discovery and validation
   - **Cell 4**: Multi-model inference (may take hours)
   - **Cell 5**: Consensus generation and postprocessing
   - **Cell 6**: Quality analysis and visualization
5. Review results in the output directories as described in the Results Summary

In [ ]:
# Setup: Import Libraries and Configure Paths

# Import required Python libraries
import os
import subprocess
import glob
from pathlib import Path

# Configuration Section
# Modify these paths according to your project setup

# BASE_DIR: Root directory of the DARE2d project
BASE_DIR = "path/to/your/project/root"  # Example: r"C:\Users\YourName\Projects\DARE2d"

# Define subdirectories relative to BASE_DIR
INPUT_DIR = os.path.join(BASE_DIR, "input")  # Directory containing input .tif images
OUTPUT_DIR = os.path.join(BASE_DIR, "output")  # Directory for saving inference results
REG_DIR = os.path.join(BASE_DIR, "regression_checkpoints")  # Regression model checkpoints
SEG_DIR = os.path.join(BASE_DIR, "segmentation_checkpoints")  # Segmentation model checkpoints

# Verify that the base directory exists
if not os.path.exists(BASE_DIR):
    raise FileNotFoundError(f"Base directory not found: {BASE_DIR}. Please update BASE_DIR to the correct path.")

print(f"Project root: {BASE_DIR}")
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print("Setup completed successfully.")

## Data Discovery

In this section, we identify all input images that will be processed. The script expects images in .tif format placed in the input directory.

The following code:
- Scans the input directory for .tif files
- Displays the number of images found
- Lists the image names for verification

In [ ]:
# Data Discovery: Find Input Images

# Discover all .tif images in the input directory
input_images = glob.glob(os.path.join(INPUT_DIR, "*.tif"))

if not input_images:
    print("Error: No .tif images found in the input directory!")
    print(f"Input directory: {INPUT_DIR}")
    print("Please ensure input images are placed in the correct directory.")
    # In a notebook, you might want to raise an error or stop execution
    raise FileNotFoundError("No input images found")

print(f"Found {len(input_images)} input images:")
for img_path in input_images:
    image_name = Path(img_path).stem
    print(f"  - {image_name}")

# Store image names for later use
image_names = [Path(img).stem for img in input_images]

## Batch Inference Execution

This section performs the core inference process. For each input image, the script runs inference using 8 different model sets (numbered 1-8). Each model set consists of a pair of regression and segmentation models trained on different data subsets.

The process for each image-model combination:
1. Locate the appropriate model checkpoints
2. Create an output directory for results
3. Execute the inference using the `multistage_detection2d` script
4. Save results including detected division centers, visualizations, and raw predictions

**Note:** This process may take significant time depending on the number of images and model sets. Each inference run processes a 3D stack of 2D images.

In [ ]:
# Batch Inference: Process All Images with Multiple Model Sets

# Number of model sets to use (1-8)
num_model_sets = 8

# Process each image
for img_path, image_name in zip(input_images, image_names):
    print(f"\n{'='*60}")
    print(f"Processing image: {image_name}")
    print(f"{'='*60}")

    # Run inference with each model set
    for n in range(1, num_model_sets + 1):
        print(f"\n--- Model Set {n} ---")

        # Construct paths to model checkpoints
        # Assumes organization: checkpoints_set_{n}_all_but_target/best.h5
        reg_checkpoint = os.path.join(REG_DIR, f"checkpoints_set_{n}_all_but_target", "best.h5")
        seg_checkpoint = os.path.join(SEG_DIR, f"checkpoints_set_{n}_all_but_target", "best.h5")

        # Verify model files exist
        if not os.path.exists(reg_checkpoint):
            print(f"Warning: Regression checkpoint not found: {reg_checkpoint}")
            continue
        if not os.path.exists(seg_checkpoint):
            print(f"Warning: Segmentation checkpoint not found: {seg_checkpoint}")
            continue

        # Create output directory for this image-model combination
        output_folder = os.path.join(OUTPUT_DIR, f"{image_name}_{n}")
        os.makedirs(output_folder, exist_ok=True)

        # Construct the inference command
        cmd = [
            "python",
            "-m", "scripts.inference.multistage_detection2d",
            "--regression", reg_checkpoint,
            "--segmentation", seg_checkpoint,
            "--img", img_path,
            "--output", output_folder
        ]

        print(f"Running command: {' '.join(cmd)}")

        try:
            # Execute the inference
            result = subprocess.run(cmd, check=True, capture_output=True, text=True)
            print(f"✓ Successfully completed inference for {image_name} with model set {n}")
            # Optionally print stdout if needed: print(result.stdout)

        except subprocess.CalledProcessError as e:
            print(f"✗ Error running inference for {image_name} with model set {n}")
            print(f"Error details: {e}")
            print(f"Stderr: {e.stderr}")
            continue

print(f"\n{'='*60}")
print("Batch inference completed!")
print(f"Results saved in: {OUTPUT_DIR}")
print("Each image has subdirectories for each model set (1-8)")
print(f"Total combinations processed: {len(input_images)} images × {num_model_sets} model sets = {len(input_images) * num_model_sets}")
print(f"{'='*60}")

## Postprocessing and Consensus Generation

After running inference with multiple models, we perform postprocessing to aggregate detections across models and generate consensus results. This step:

1. **Loads detections** from all model outputs for each image
2. **Spatial clustering** using HDBSCAN (or DBSCAN fallback) to group nearby detections
3. **Consensus aggregation** per cluster: median positions, chosen angles, statistics
4. **Temporal deduplication** across consecutive frames
5. **Visualization** with uncertainty wedges and quality metrics
6. **Quality scoring** based on model agreement and uncertainty measures

The postprocessing script expects the output structure created by the inference step and produces:
- Consensus division detections in `chosen_divisions/` subfolder
- Visualized results in `post_processed_{image_name}.tiff`
- Summary statistics in `post_processed_{image_name}_summary.csv`

In [ ]:
# Postprocessing: Aggregate Multi-Model Results

# Create directory for postprocessing outputs
POSTPROCESS_DIR = os.path.join(BASE_DIR, "postprocessed_results")
os.makedirs(POSTPROCESS_DIR, exist_ok=True)

# Postprocessing parameters
eps = 10  # Clustering epsilon (spatial distance threshold)
min_models = 6  # Minimum number of models required for consensus
num_models = 8  # Total number of models
angle_mode = "auto"  # Angle selection mode

print(f"Postprocessing outputs will be saved to: {POSTPROCESS_DIR}")

# Process each image with postprocessing
for img_path, image_name in zip(input_images, image_names):
    print(f"\n{'='*60}")
    print(f"Postprocessing image: {image_name}")
    print(f"{'='*60}")

    # Define output directory for this image's postprocessed results
    image_postprocess_dir = os.path.join(POSTPROCESS_DIR, image_name)
    os.makedirs(image_postprocess_dir, exist_ok=True)

    # Construct postprocessing command
    cmd = [
        "python",
        "scripts/postprocessing/main.py",
        "--output_root", OUTPUT_DIR,
        "--image_name", image_name,
        "--image_stack", img_path,
        "--save_dir", image_postprocess_dir,
        "--eps", str(eps),
        "--min_models", str(min_models),
        "--num_models", str(num_models),
        "--angle_mode", angle_mode
    ]

    print(f"Running postprocessing command: {' '.join(cmd)}")

    try:
        # Execute postprocessing
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"✓ Successfully completed postprocessing for {image_name}")
        # Uncomment to see detailed output: print(result.stdout)

    except subprocess.CalledProcessError as e:
        print(f"✗ Error in postprocessing for {image_name}")
        print(f"Error details: {e}")
        print(f"Stderr: {e.stderr}")
        continue

print(f"\n{'='*60}")
print("Postprocessing completed for all images!")
print(f"Results saved in: {POSTPROCESS_DIR}")
print("Each image has its own subfolder with consensus results.")
print(f"{'='*60}")

## Quality Analysis and Visualization

After postprocessing, we can generate quality metrics and visualizations to assess the reliability of the consensus detections. This includes:

- **Distribution plots** for division lengths, number of supporting models, and support fractions
- **Quality scores** computed from model agreement and uncertainty measures
- **Timecourse plots** showing division events across frames with error bars
- **Summary statistics** and metrics CSV files

The plots script analyzes the postprocessed summary CSV and generates comprehensive visualizations.

In [ ]:
# Quality Analysis and Visualization

# Create directory for plots and analysis
PLOTS_DIR = os.path.join(BASE_DIR, "analysis_plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f"Analysis plots will be saved to: {PLOTS_DIR}")

# Generate plots for each postprocessed image
for image_name in image_names:
    image_postprocess_dir = os.path.join(POSTPROCESS_DIR, image_name)

    # Check if postprocessing was successful for this image
    summary_csv = os.path.join(image_postprocess_dir, f"post_processed_{image_name}_summary.csv")
    processed_tiff = os.path.join(image_postprocess_dir, f"post_processed_{image_name}.tiff")

    if not os.path.exists(summary_csv):
        print(f"Warning: Summary CSV not found for {image_name}, skipping plots")
        continue

    if not os.path.exists(processed_tiff):
        print(f"Warning: Processed TIFF not found for {image_name}, skipping plots")
        continue

    print(f"\nGenerating plots for {image_name}...")

    # Create plots subdirectory for this image
    image_plots_dir = os.path.join(PLOTS_DIR, image_name)
    os.makedirs(image_plots_dir, exist_ok=True)

    # Construct plots command
    cmd = [
        "python",
        "scripts/postprocessing/plots.py",
        "--summary", summary_csv,
        "--tiff", processed_tiff,
        "--plots_dir", image_plots_dir
    ]

    print(f"Running plots command: {' '.join(cmd)}")

    try:
        # Execute plots generation
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"✓ Successfully generated plots for {image_name}")

    except subprocess.CalledProcessError as e:
        print(f"✗ Error generating plots for {image_name}")
        print(f"Error details: {e}")
        print(f"Stderr: {e.stderr}")
        continue

print(f"\n{'='*60}")
print("Quality analysis and visualization completed!")
print(f"Plots and metrics saved in: {PLOTS_DIR}")
print("Each image has its own subfolder with analysis results.")
print(f"{'='*60}")

## Complete Pipeline Results Summary

### Pipeline Overview
This notebook implements a complete 2D cell division detection pipeline:

1. **Multi-Model Inference**: Runs 8 different model pairs on each input image
2. **Consensus Generation**: Aggregates detections using spatial clustering and temporal deduplication
3. **Quality Assessment**: Computes metrics and generates visualizations for result evaluation

### Output Directory Structure

**Root Directories:**
- `OUTPUT_DIR/` (inference results)
- `POSTPROCESS_DIR/` (consensus results)
- `PLOTS_DIR/` (analysis and plots)

**Per-Image Organization:**
```
project_root/
├── output/
│   ├── {image_name}_1/{image_name}/
│   │   ├── division_position1.npy, division_position2.npy, ...
│   │   ├── {image_name}.tiff (input copy)
│   │   ├── {image_name}_result.tiff (inference visualization)
│   │   └── {image_name}_raw.tiff (segmentation predictions)
│   ├── {image_name}_2/{image_name}/
│   └── ...
├── postprocessed_results/
│   └── {image_name}/
│       ├── chosen_divisions/
│       │   └── division_position{frame}.npy (consensus detections)
│       ├── post_processed_{image_name}.tiff (final visualization)
│       └── post_processed_{image_name}_summary.csv (statistics)
└── analysis_plots/
    └── {image_name}/
        ├── distribution_plots.png
        ├── quality_scores.png
        ├── timecourse_plots.png
        └── metrics.csv
```

### Key Output Files

**Inference Results (per model):**
- `division_position{i}.npy`: Raw detections for frame i
- `{image_name}_result.tiff`: Visualization with detected divisions
- `{image_name}_raw.tiff`: Segmentation probability maps

**Consensus Results:**
- `chosen_divisions/division_position{i}.npy`: Final consensus detections
- `post_processed_{image_name}.tiff`: Consensus visualization with uncertainty wedges
- `post_processed_{image_name}_summary.csv`: Comprehensive statistics

**Analysis Results:**
- Distribution plots for division lengths, model support, angular uncertainty
- Quality score plots and rankings
- Timecourse plots showing division events across frames
- Summary metrics CSV with evaluation statistics

### Quality Metrics

The pipeline provides several quality indicators:

- **Support Fraction**: Percentage of models agreeing on a detection
- **Positional Standard Deviation**: Spatial uncertainty of consensus position
- **Angular Standard Deviation**: Orientation uncertainty
- **Length Statistics**: Mean, std, min, max division lengths
- **Quality Score**: Composite metric combining all uncertainty measures

### Usage Instructions

1. **Setup**: Update `BASE_DIR` and verify all paths
2. **Data**: Place .tif image stacks in `INPUT_DIR`
3. **Models**: Ensure checkpoints are in `REG_DIR` and `SEG_DIR`
4. **Execution**: Run cells sequentially (may take hours for large datasets)
5. **Results**: Review outputs in the three main directories

### Expected Processing Time

- **Small dataset** (5 images): ~30-60 minutes
- **Medium dataset** (20 images): ~2-4 hours
- **Large dataset** (100+ images): ~10-20 hours

Processing time depends on image size, number of frames, and available hardware.

### Troubleshooting Guide

**Common Issues:**
- **"No .tif images found"**: Check `INPUT_DIR` path and file formats
- **"Model checkpoint not found"**: Verify `REG_DIR`/`SEG_DIR` paths and checkpoint naming
- **Memory errors**: Reduce batch sizes or process fewer images
- **Postprocessing fails**: Ensure inference completed successfully for all models

**Performance Tips:**
- Use SSD storage for faster I/O
- Process images in smaller batches if memory is limited
- Monitor disk space (outputs can be 10-50x input size)

### Next Steps and Applications

**Analysis Applications:**
1. **Quantitative Evaluation**: Compare against ground truth annotations
2. **Model Comparison**: Analyze performance across different training sets
3. **Parameter Optimization**: Tune clustering and consensus parameters
4. **Biological Insights**: Study division patterns and kinetics

**Further Processing:**
- Load consensus detections for downstream analysis
- Use quality scores to filter high-confidence events
- Integrate with tracking algorithms for cell lineage analysis
- Apply to new datasets with the trained models

### Citation and References

If using this pipeline in your research, please cite:

[Your Name et al. "Multi-Model Consensus Framework for Robust 2D Cell Division Detection." Journal Name, Year.]

**Software Dependencies:**
- DARE2d framework
- Python 3.8+
- TensorFlow/Keras
- scikit-learn, scikit-image
- matplotlib, numpy, pandas
- hdbscan (optional, falls back to DBSCAN)

**Contact:** [Your email] for questions or collaboration opportunities.